# pig-math TTS Phase 1B — 全量生成（YunJhe）

User 試聽後選定 **zh-TW-YunJheNeural**（沉穩男聲）。這份 notebook 一次性
生成 ~200 個 MP3 涵蓋整站常用語音，Phase 2 webapp 端接這些檔案。

## 涵蓋範圍（從 codebase 抽出）

| 類別 | 數量 | 例子 |
|---|---|---|
| 數字 zh 0-100 | 101 | 「七」「二十一」「一百」 |
| 數字 en 0-9 | 10 | "five" "nine"（practice page） |
| 運算詞 | 10 | 加 / 減 / 乘以 / 除以 / 等於 / 和 / 哪個比較大 / 還是 / 是 / 不是 |
| 題號 | 20 | 第一題～第二十題 |
| 答對鼓勵 (zh) | 6 | 答對了！太厲害了！好棒喔！... |
| 答錯鼓勵 (zh) | 5 | 沒關係再試試！差一點點！... |
| 答對 (en) | 6 | Great job! Amazing! Perfect! ... |
| 答錯 (en) | 5 | Try again! Almost! ... |
| 連勝里程碑 | 5 | 連續 3 題全對！... 20 連勝！ |
| 中場休息 | 6 | 辛苦囉、記得喝口水喔 ... |
| 描紅引導 | 12 | 「我們來練習寫零～九」+ 「沿著虛線描寫看看」 |
| 問句結尾 | 2 | 「等於多少？」「等於幾？」|

**合計約 195 個 MP3，預期 < 8 MB**。

In [1]:
!pip install -q edge-tts

## Step 1：phrase 清單

每筆 `(key, text, voice)`。key 對應 Phase 2 webapp 端的 phrase id。

In [2]:
ZH_VOICE = 'zh-TW-YunJheNeural'
EN_VOICE = 'en-US-AriaNeural'

# 數字轉中文（與 src/lib/audio/speech.ts 的 numberToChinese 對齊）
DIGITS = ['零','一','二','三','四','五','六','七','八','九']
def num_to_zh(n: int) -> str:
    if n < 10: return DIGITS[n]
    if n == 10: return '十'
    if n < 20: return '十' + DIGITS[n - 10]
    if n < 100:
        t, o = divmod(n, 10)
        return DIGITS[t] + '十' + (DIGITS[o] if o else '')
    if n == 100: return '一百'
    # 100-999 給 Phase 2 用，這裡先不預生成
    raise ValueError(f'unsupported: {n}')

DIGITS_EN = ['zero','one','two','three','four','five','six','seven','eight','nine']

# (key, text, voice) tuples
phrases = []

# 1. 數字 zh 0-100
for n in range(0, 101):
    phrases.append((f'num_zh_{n}', num_to_zh(n), ZH_VOICE))

# 2. 數字 en 0-9（practice page 用）
for n in range(0, 10):
    phrases.append((f'num_en_{n}', DIGITS_EN[n], EN_VOICE))

# 3. 運算詞 / 連接詞
ops = [
    ('op_add',           '加'),
    ('op_sub',           '減'),
    ('op_mul',           '乘以'),
    ('op_div',           '除以'),
    ('op_equals',        '等於'),
    ('op_and',           '和'),
    ('op_which_bigger',  '哪個比較大'),
    ('op_or',            '還是'),
    ('op_is',            '是'),
    ('op_is_not',        '不是'),
]
for k, t in ops:
    phrases.append((k, t, ZH_VOICE))

# 4. 題號 第一題 ~ 第二十題
for n in range(1, 21):
    phrases.append((f'problem_n_{n}', f'第{num_to_zh(n)}題', ZH_VOICE))

# 5. 答對鼓勵 zh（對應 FeedbackOverlay.CORRECT_PHRASES）
correct_zh = [
    ('feedback_correct_1', '答對了！'),
    ('feedback_correct_2', '太厲害了！'),
    ('feedback_correct_3', '好棒喔！'),
    ('feedback_correct_4', '完全正確！'),
    ('feedback_correct_5', '你好聰明！'),
    ('feedback_correct_6', '太強了！'),
]
for k, t in correct_zh: phrases.append((k, t, ZH_VOICE))

# 6. 答錯鼓勵 zh（對應 FeedbackOverlay.WRONG_PHRASES）
wrong_zh = [
    ('feedback_wrong_1', '沒關係，再試試！'),
    ('feedback_wrong_2', '差一點點！'),
    ('feedback_wrong_3', '加油，你可以的！'),
    ('feedback_wrong_4', '想一想再試！'),
    ('feedback_wrong_5', '慢慢來沒關係！'),
]
for k, t in wrong_zh: phrases.append((k, t, ZH_VOICE))

# 7. 答對 en
correct_en = [
    ('feedback_correct_en_1', 'Great job!'),
    ('feedback_correct_en_2', 'Amazing!'),
    ('feedback_correct_en_3', 'Awesome!'),
    ('feedback_correct_en_4', 'Perfect!'),
    ('feedback_correct_en_5', 'So smart!'),
    ('feedback_correct_en_6', 'Wonderful!'),
]
for k, t in correct_en: phrases.append((k, t, EN_VOICE))

# 8. 答錯 en
wrong_en = [
    ('feedback_wrong_en_1', 'Try again!'),
    ('feedback_wrong_en_2', 'Almost!'),
    ('feedback_wrong_en_3', 'You can do it!'),
    ('feedback_wrong_en_4', 'Think again!'),
    ('feedback_wrong_en_5', 'Take your time!'),
]
for k, t in wrong_en: phrases.append((k, t, EN_VOICE))

# 9. 連勝里程碑
streak = [
    ('streak_3',  '連續三題全對！繼續保持！'),
    ('streak_5',  '哇！連續五題全對，超級厲害！'),
    ('streak_10', '十連勝！你是數學小天才！'),
    ('streak_15', '十五連勝！不可思議的表現！'),
    ('streak_20', '二十連勝！你已經是數學大師了！'),
]
for k, t in streak: phrases.append((k, t, ZH_VOICE))

# 10. 中場休息（對應 BreakReminder.BREAK_VOICE_LINES）
breaks = [
    ('break_1', '辛苦囉，休息一下，記得喝口水喔'),
    ('break_2', '做得很棒，站起來動一動'),
    ('break_3', '讓眼睛休息一下，看看遠遠的地方'),
    ('break_4', '深呼吸三次，放輕鬆'),
    ('break_5', '喝口水，伸伸懶腰'),
    ('break_6', '已經很努力了，放鬆一下吧'),
]
for k, t in breaks: phrases.append((k, t, ZH_VOICE))

# 11. 描紅引導
for n in range(0, 10):
    phrases.append((f'trace_zh_{n}', f'我們來練習寫{DIGITS[n]}', ZH_VOICE))
phrases.append(('trace_intro',  '沿著虛線描寫看看', ZH_VOICE))
phrases.append(('trace_great',  '寫得很棒！', ZH_VOICE))

# 12. 問句結尾
phrases.append(('q_equals_what',  '等於多少？', ZH_VOICE))
phrases.append(('q_equals_how',   '等於幾？', ZH_VOICE))

print(f'合計 {len(phrases)} 個 phrase')
# 顯示分佈
from collections import Counter
voice_counts = Counter(v for _, _, v in phrases)
for v, c in voice_counts.items():
    print(f'  {v}: {c}')

合計 188 個 phrase
  zh-TW-YunJheNeural: 167
  en-US-AriaNeural: 21


## Step 2：批次生成 MP3

rate=-10% 比預設稍慢、給小孩聽更舒服。所有 MP3 平鋪在 `audio-tts/<key>.mp3`，
搭配同層 `manifest.json` 提供 webapp 載入用。

In [3]:
import asyncio
import json
import os
import edge_tts

OUTPUT_DIR = 'audio-tts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 每個 generate() 最多花的秒數（避免某次 request 永遠不回應）
PER_TASK_TIMEOUT_S = 20
MAX_RETRIES = 3
RETRY_DELAY_S = 1.0
# 微軟免費 TTS 接口對並行很敏感；BATCH=4 + batch 間 sleep 0.8s
BATCH = 4
BATCH_SLEEP_S = 0.8

async def _save_with_timeout(text, voice, out_path):
    communicate = edge_tts.Communicate(text, voice, rate='-10%')
    # wait_for 包起來：超時就 cancel
    await asyncio.wait_for(communicate.save(out_path), timeout=PER_TASK_TIMEOUT_S)

async def generate(key: str, text: str, voice: str):
    out_path = os.path.join(OUTPUT_DIR, f'{key}.mp3')
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            await _save_with_timeout(text, voice, out_path)
            if os.path.getsize(out_path) < 100:
                raise RuntimeError('MP3 太小')
            return (key, True, None)
        except asyncio.TimeoutError:
            last_err = f'timeout（{PER_TASK_TIMEOUT_S}s）'
            # timeout 後刪掉 partial 檔
            if os.path.exists(out_path):
                try: os.remove(out_path)
                except OSError: pass
            await asyncio.sleep(RETRY_DELAY_S)
        except Exception as e:
            last_err = str(e)
            await asyncio.sleep(RETRY_DELAY_S)
    return (key, False, last_err)

async def main():
    failed = []
    done = 0
    for i in range(0, len(phrases), BATCH):
        chunk = phrases[i:i + BATCH]
        results = await asyncio.gather(*[generate(k, t, v) for k, t, v in chunk])
        for key, ok, err in results:
            if not ok:
                failed.append((key, err))
        done += len(chunk)
        # 顯示當前 batch 哪些失敗（讓你知道是哪些卡住）
        bad = [r[0] for r in results if not r[1]]
        suffix = f'（失敗 {len(failed)}：{bad}）' if bad else f'（失敗 {len(failed)}）'
        print(f'  {done}/{len(phrases)} 完成{suffix}', flush=True)
        await asyncio.sleep(BATCH_SLEEP_S)
    print()
    if failed:
        print(f'⚠ 有 {len(failed)} 個 phrase 失敗，再跑一次這個 cell 補生成：')
        for key, err in failed[:15]:
            print(f'    {key}: {err}')
        if len(failed) > 15:
            print(f'    ... 還有 {len(failed) - 15} 個')
    else:
        print(f'✓ 全部 {len(phrases)} 個 MP3 生成完成')

# 跑前先把已生成的算進去（重跑補失敗用）
SKIP_EXISTING = True
if SKIP_EXISTING:
    before = sum(
        1 for k, _, _ in phrases
        if os.path.exists(os.path.join(OUTPUT_DIR, f'{k}.mp3'))
        and os.path.getsize(os.path.join(OUTPUT_DIR, f'{k}.mp3')) >= 100
    )
    if before > 0:
        print(f'（已有 {before} 個 MP3，重跑時只補沒生成的）')
    phrases_to_run = [
        (k, t, v) for k, t, v in phrases
        if not (
            os.path.exists(os.path.join(OUTPUT_DIR, f'{k}.mp3'))
            and os.path.getsize(os.path.join(OUTPUT_DIR, f'{k}.mp3')) >= 100
        )
    ]
    if phrases_to_run:
        phrases_backup = phrases
        phrases = phrases_to_run
        try:
            await main()
        finally:
            phrases = phrases_backup
    else:
        print('✓ 所有 MP3 都已存在，跳過生成')
else:
    await main()

  4/188 完成（失敗 0）
  8/188 完成（失敗 0）
  12/188 完成（失敗 0）
  16/188 完成（失敗 0）
  20/188 完成（失敗 0）
  24/188 完成（失敗 0）
  28/188 完成（失敗 0）
  32/188 完成（失敗 0）
  36/188 完成（失敗 0）
  40/188 完成（失敗 0）
  44/188 完成（失敗 0）
  48/188 完成（失敗 0）
  52/188 完成（失敗 0）
  56/188 完成（失敗 0）
  60/188 完成（失敗 0）
  64/188 完成（失敗 0）
  68/188 完成（失敗 0）
  72/188 完成（失敗 0）
  76/188 完成（失敗 0）
  80/188 完成（失敗 0）
  84/188 完成（失敗 0）
  88/188 完成（失敗 0）
  92/188 完成（失敗 0）
  96/188 完成（失敗 0）
  100/188 完成（失敗 0）
  104/188 完成（失敗 0）
  108/188 完成（失敗 0）
  112/188 完成（失敗 0）
  116/188 完成（失敗 0）
  120/188 完成（失敗 0）
  124/188 完成（失敗 0）
  128/188 完成（失敗 0）
  132/188 完成（失敗 0）
  136/188 完成（失敗 0）
  140/188 完成（失敗 0）
  144/188 完成（失敗 0）
  148/188 完成（失敗 0）
  152/188 完成（失敗 0）
  156/188 完成（失敗 0）
  160/188 完成（失敗 0）
  164/188 完成（失敗 0）
  168/188 完成（失敗 0）
  172/188 完成（失敗 0）
  176/188 完成（失敗 0）
  180/188 完成（失敗 0）
  184/188 完成（失敗 0）
  188/188 完成（失敗 0）

✓ 全部 188 個 MP3 生成完成


## Step 3：產生 manifest.json

Phase 2 webapp 載入 manifest 把 phrase key 對應到 MP3 URL。

In [4]:
manifest = {
    'voice': {'zh': ZH_VOICE, 'en': EN_VOICE},
    'rate': '-10%',
    'phrases': {k: t for k, t, _ in phrases},
    'count': len(phrases),
}

with open(os.path.join(OUTPUT_DIR, 'manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(f'✓ manifest.json（{len(manifest["phrases"])} entries）')

✓ manifest.json（188 entries）


## Step 4：打包 + 拿下載 URL（0x0.st，24h）

In [5]:
import shutil
import subprocess

zip_path = shutil.make_archive('audio-tts', 'zip', '.', OUTPUT_DIR)
size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f'✓ 打包：{zip_path}（{size_mb:.1f} MB）')

print()
print('▶ 上傳到 0x0.st ...')
r = subprocess.run(
    ['curl', '-s', '-F', f'file=@{zip_path}', '-F', 'expires=1', 'https://0x0.st'],
    capture_output=True, text=True, timeout=120,
)
url = r.stdout.strip() if r.returncode == 0 else None
if url and url.startswith('http'):
    print()
    print('═══════════════════════════════════════════════════════════')
    print(f'  ⬇  下載連結（24h 內有效）:')
    print(f'      {url}')
    print('═══════════════════════════════════════════════════════════')
    print()
    print('▼ 在本機 Math/ 跑（會自動下載 + 解壓進 public/audio/tts/）')
    print('  mkdir -p ~/Downloads/pig-math-public/audio/tts')
    print('  cd ~/Downloads/pig-math-public/audio/tts')
    print(f'  curl -sLO {url}')
    print(f'  mv $(basename {url}) audio-tts.zip')
    print('  unzip -o audio-tts.zip && rm audio-tts.zip')
    print()
    print('  跑完跟 Claude 說「audio 已就位」，他繼續 Phase 2 整合')
else:
    print('✗ 上傳失敗，請手動從 Colab 介面下載 audio-tts.zip')
    print(f'  位置: {zip_path}')

✓ 打包：/content/audio-tts.zip（1.4 MB）

▶ 上傳到 0x0.st ...
✗ 上傳失敗，請手動從 Colab 介面下載 audio-tts.zip
  位置: /content/audio-tts.zip
